# Notebook 04 — Chatbot Completo Integrado

**Curso:** Minería de Textos | **Proyecto 3** | CUC

---



In [1]:
import sys
sys.path.append('../app')
sys.path.append('../src')

import json
from pathlib import Path

from src.mongo_utils import mongo_utils
from src.finetuning_utils import finetuning_utils
from src.rag_utils import rag_utils
from src.chatbot_engine import chatbot_engine

print('Librerias cargadas.')

C:\Users\pmari\OneDrive\Para Revisar\Documentos\Pablo\Cuc\2026\Mineria de Textos\chatbot-musical-inteligente\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Librerias cargadas.


## 1. Inicializar sistema completo

In [2]:
db_conexion = mongo_utils()
finetuning = finetuning_utils()
rag = rag_utils()

if db_conexion.verificar_conexion():
   canciones_raw=db_conexion.cargar_canciones()
   corpus_etiquetado = finetuning.etiquetar_corpus_con_modelo(canciones_raw)
   print(f'Canciones con emocion: {len(corpus_etiquetado)}')
   # Verificar que tienen el campo emocion
   ejemplo = corpus_etiquetado[0]
   print(f'Ejemplo: {ejemplo.get("titulo")} — Emocion: {ejemplo.get("emocion")} ({ejemplo.get("emocion_score",0):.0%})')
   rag.inicializar(corpus_etiquetado)
   bot = chatbot_engine()
   print('\nSistema listo: Fine-Tuning + RAG integrados.')
else:
   canciones_raw=[]
   print(f'Canciones no cargadas: {len(canciones_raw)}')



2026-04-20 00:22:59 | INFO     | mongo_utils | Conectando a MongoDB Atlas | DB: analisisMusical | Col: analisisMusical
2026-04-20 00:23:00 | INFO     | mongo_utils | Conexión verificada correctamente.
2026-04-20 00:23:00 | INFO     | mongo_utils | Cargando canciones | limite=None | solo_con_letra=True
2026-04-20 00:23:02 | INFO     | mongo_utils | Canciones cargadas: 6940
2026-04-20 00:23:02 | INFO     | finetuning_utils | Cargando corpus etiquetado desde caché...
2026-04-20 00:23:02 | INFO     | finetuning_utils | 6940 canciones cargadas con emoción.
Canciones con emocion: 6940
Ejemplo: ​thank u, next — Emocion: amor (97%)
2026-04-20 00:23:02 | INFO     | rag_utils | Caché completo encontrado. Cargando RAG desde disco...
2026-04-20 00:23:02 | INFO     | rag_utils | Cargando embeddings desde caché...
2026-04-20 00:23:02 | INFO     | rag_utils | Embeddings cargados: (6940, 384) | Chunks: 6940
2026-04-20 00:23:02 | INFO     | rag_utils | Cargando índice FAISS desde caché...
2026-04-20 00

Loading weights: 100%|██████████| 282/282 [00:00<00:00, 1870.54it/s]
The tied weights mapping and config for this model specifies to tie shared.weight to lm_head.weight, but both are present in the checkpoints with different values, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning.


2026-04-20 00:23:10 | INFO     | chatbot_engine | Flan-T5 cargado correctamente.

Sistema listo: Fine-Tuning + RAG integrados.


## 2. Prueba interactiva con emocion detectada

In [3]:
def chat(pregunta):
    resultado = bot.responder(pregunta)
    emocion = resultado.get('emocion')
    print(f'Usuario: {pregunta}')
    if emocion:
        print(f'[Emocion detectada: {emocion["emocion"]} ({emocion["score"]:.0%})]')
    print(f'Bot: {resultado["respuesta"]}')
    print(f'[Fuentes: {", ".join(c["titulo"]+" ("+c["emocion"]+")" for c in resultado["chunks"][:3])}]')
    print()
    return resultado

# Conversacion de prueba con memoria
chat('Estoy triste, que cancion me recomiendas?')
chat('Dame otra similar.')
chat('De que artista es la primera que mencionaste?')

2026-04-20 00:23:44 | INFO     | chatbot_engine | Pregunta recibida: 'Estoy triste, que cancion me recomiendas?'


Loading weights: 100%|██████████| 104/104 [00:00<00:00, 3665.73it/s]


2026-04-20 00:23:47 | INFO     | finetuning_utils | Clasificador cargado para inferencia.
2026-04-20 00:23:47 | INFO     | rag_utils | Cargando modelo de embeddings: sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2


Loading weights: 100%|██████████| 199/199 [00:00<00:00, 3275.41it/s]
BertModel LOAD REPORT from: sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


2026-04-20 00:23:54 | INFO     | rag_utils | Modelo cargado. Dimensión: 384


C:\Users\pmari\OneDrive\Para Revisar\Documentos\GitHub\chatbot-musical-inteligente\src\rag_utils.py:165: FutureWarning: The `get_sentence_embedding_dimension` method has been renamed to `get_embedding_dimension`.
  dim = self._modelo_emb.get_sentence_embedding_dimension()


2026-04-20 00:23:57 | INFO     | chatbot_engine | Respuesta generada (24 chars)
Usuario: Estoy triste, que cancion me recomiendas?
[Emocion detectada: alegria (60%)]
Bot: no hay nada en el ttulo?
[Fuentes: A Place In This World (Live from SoHo) (alegria), I Forgot That You Existed (Piano/Vocal) (alegria), Your Song (alegria)]

2026-04-20 00:23:57 | INFO     | chatbot_engine | Pregunta recibida: 'Dame otra similar.'
2026-04-20 00:23:58 | INFO     | chatbot_engine | Respuesta generada (9 chars)
Usuario: Dame otra similar.
[Emocion detectada: alegria (31%)]
Bot: i get her
[Fuentes: Do It Now (alegria), Cheers (Drink to That) (alegria), White Iverson (alegria)]

2026-04-20 00:23:58 | INFO     | chatbot_engine | Pregunta recibida: 'De que artista es la primera que mencionaste?'
2026-04-20 00:24:08 | INFO     | chatbot_engine | Respuesta generada (748 chars)
Usuario: De que artista es la primera que mencionaste?
[Emocion detectada: alegria (28%)]
Bot: oh yeah and i ain't just talking about y

{'respuesta': "oh yeah and i ain't just talking about your face but look at your face oh yeah and i'm so curious your mind got me feeling some type of way what can i say you're gorgeous huh huh ohhuh two weeks later you should take it as a compliment that i got drunk and made fun of the way you talk i tell 'em a little bit later you should take it as a compliment that i got drunk and made fun of the way you talk 'cause you're cool yeah you should think about the consequence of your magnetic field getting way too strong you're so cool yeah later that same day you should take it as a compliment that i got drunk and made fun of the way you talk ohhuh ohhuh you should think about the consequence of your magnetic field being a little too strong huh huh ohhuh",
 'chunks': [{'texto': "first version you're so gorgeous and i ain't just talking about your face but look at your face oh yeah and i'm so curious your mind got me feeling some type of way what can i say you're gorgeous huh huh ohhuh t

## 3. Generar 10 conversaciones de prueba

In [4]:
bot.limpiar_historial()

conversaciones = [
    'Que canciones hablan de desamor?',
    'Que artistas de pop hay en el corpus?',
    'Estoy muy feliz hoy, que cancion me recomiendas?',
    'Necesito algo para llorar un rato.',
    'Que diferencia hay entre una balada triste y una alegre?',
    'Dame otra cancion parecida a la ultima que mencionaste.',
    'Y esa de quien es?',
    'Dame una cancion de reggaeton con emocion de amor.',
    'Cuanto cuesta un vuelo a Madrid?',       # fuera de dominio
    'Quien gano el mundial de futbol en 2022?', # fuera de dominio
]

registro = []
bot.limpiar_historial()

for i, pregunta in enumerate(conversaciones, 1):
    print(f'--- Conversacion {i} ---')
    resultado = chat(pregunta)
    registro.append({
        'id': i, 'pregunta': pregunta, 'respuesta': resultado['respuesta'],
        'emocion_detectada': resultado.get('emocion'),
        'fuentes': [{'titulo': c['titulo'], 'artista': c['artista'],
                     'emocion': c['emocion'], 'score': c['score']}
                    for c in resultado['chunks']],
    })

# Guardar en metricas.json
Path('../resultados').mkdir(exist_ok=True)
metricas_path = Path('../resultados/metricas.json')
metricas_existentes = {}
if metricas_path.exists():
    with open(metricas_path, 'r', encoding='utf-8') as f:
        metricas_existentes = json.load(f)
metricas_existentes['conversaciones_prueba'] = registro
with open(metricas_path, 'w', encoding='utf-8') as f:
    json.dump(metricas_existentes, f, ensure_ascii=False, indent=2)
print('10 conversaciones guardadas en resultados/metricas.json')

2026-04-20 00:25:14 | INFO     | chatbot_engine | Historial conversacional limpiado.
2026-04-20 00:25:14 | INFO     | chatbot_engine | Historial conversacional limpiado.
--- Conversacion 1 ---
2026-04-20 00:25:14 | INFO     | chatbot_engine | Pregunta recibida: 'Que canciones hablan de desamor?'
2026-04-20 00:25:28 | INFO     | chatbot_engine | Respuesta generada (712 chars)
Usuario: Que canciones hablan de desamor?
[Emocion detectada: nostalgia (40%)]
Bot: oh yeah i don't wanna be just a memory baby yeah hoo hoo hoo hoo hoo hoo hoo hoo hoo hoo hoo hoo hoo hoo hoo hoo hoo hoo hoo hoo hoo hoo hoo hoo hoo hoo hoo hoo hoo hoo hoo hoo hoo hoo hoo hoo hoo hoo hoo hoo hoo hoo hoo hoo hoo hoo hoo hoo hoo hoo hoo hoo hoo hoo hoo hoo hoo hoo hoo hoo hoo hoo hoo hoo hoo hoo hoo hoo hoo hoo hoo hoo hoo hoo hoo hoo hoo hoo hoo hoo hoo hoo hoo hoo hoo hoo hoo hoo hoo hoo hoo hoo hoo hoo hoo hoo hoo hoo hoo hoo hoo hoo hoo hoo hoo hoo hoo hoo hoo hoo hoo hoo hoo hoo hoo hoo hoo hoo hoo hoo hoo hoo h

You seem to be using the pipelines sequentially on GPU. In order to maximize efficiency please use a dataset


2026-04-20 00:26:28 | INFO     | chatbot_engine | Respuesta generada (300 chars)
Usuario: Dame una cancion de reggaeton con emocion de amor.
[Emocion detectada: amor (75%)]
Bot: 94 in my hand one more time 'fore i go body lockin' put a hold on me justin bieber you know i like that you're wild so make your way over here right now we only got a little while just thinkin' out loud celebrate life with me babe don't be in a rush to go now oh girl justin bieber i need a one dance
[Fuentes: One Dance (Remix) (amor), So Amazing (amor), A Message (amor)]

--- Conversacion 9 ---
2026-04-20 00:26:28 | INFO     | chatbot_engine | Pregunta recibida: 'Cuanto cuesta un vuelo a Madrid?'
2026-04-20 00:26:30 | INFO     | chatbot_engine | Respuesta generada (35 chars)
Usuario: Cuanto cuesta un vuelo a Madrid?
[Emocion detectada: nostalgia (47%)]
Bot: Vuelve otra ve', aunque sea una ve'
[Fuentes: Madrid (nostalgia), Ballin’ Uncontrollably (nostalgia), Takin’ Back My Love (nostalgia)]

--- Conversacion 10 

## 4. Prueba de memoria conversacional

In [5]:
bot.limpiar_historial()
print('=== PRUEBA DE MEMORIA ===')
chat('Dame una cancion triste de pop.')
chat('Puedes decirme mas sobre esa cancion?')
chat('Del mismo artista hay algo mas alegre?')
print(f'Turnos en memoria: {len(bot.historial)}')

2026-04-20 00:27:08 | INFO     | chatbot_engine | Historial conversacional limpiado.
=== PRUEBA DE MEMORIA ===
2026-04-20 00:27:08 | INFO     | chatbot_engine | Pregunta recibida: 'Dame una cancion triste de pop.'
2026-04-20 00:27:23 | INFO     | chatbot_engine | Respuesta generada (869 chars)
Usuario: Dame una cancion triste de pop.
[Emocion detectada: alegria (47%)]
Bot: Put the Percs down and picked Put the Percs down and picked Put the Percs down and picked Put the Percs down and picked Put the Percs down and picked Put the Percs down and picked Put the Percs down and picked Put the Percs down and picked Put the Percs down and picked Put the Percs down and picked Put the Percs down and picked Put the Percs down and picked Put the Percs down and picked Put the Percs down and picked Put the Percs down and picked Put the Percs down and picked Put the Percs down and picked Put the Percs down and picked Put the Percs down and picked Put the Percs down and picked Put the Percs down and p